# Testing Data Baru

Notebook ini melakukan preprocessing data testing dengan aturan yang sama seperti training, lalu menghitung membership FCM menggunakan pusat klaster final hasil training.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

ROOT_DIR = Path.cwd()
DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "outputs"
TESTING_DIR = OUTPUT_DIR / "testing"
TESTING_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"

RAW_TEST_PATH = DATA_DIR / "data_testing_contoh.csv"
FEATURE_COLUMNS = [
    "Usia Ibu",
    "Usia Kehamilan",
    "Gravida",
    "Tekanan Darah",
    "Denyut Jantung Janin",
    "Nilai AFI",
    "Kejadian KPD",
]
COLUMN_ALIASES = {
    "Kejadian Ketuban Pecah Dini": "Kejadian KPD",
    "Kejadian kpd": "Kejadian KPD",
    "Nomor Rekam Medis": "Nomor RM",
}
CLUSTER_LABELS = ["Mild", "Moderate", "Severe"]

In [2]:
def clean_text(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return ""
    return text




def remove_mow_text(value):
    text = clean_text(value)
    if not text:
        return ""
    parts = [part.strip() for part in text.replace("+", ",").split(",")]
    parts = [part for part in parts if part and part.lower() != "mow"]
    return ", ".join(parts)
def parse_numeric(value):
    text = clean_text(value)
    if not text:
        return None
    text = text.replace(",", ".")
    try:
        return float(text)
    except ValueError:
        return None


def normalize_rm(value):
    text = clean_text(value)
    if not text:
        return ""
    compact = text.replace(" ", "")
    numeric = compact.replace(".0", "")
    if numeric.isdigit():
        return numeric.zfill(6)
    return compact


def parse_range_numeric(value):
    text = clean_text(value)
    if not text:
        return None
    text = text.replace("–", "-").replace("—", "-").replace(" ", "")
    if "-" in text:
        parts = [part for part in text.split("-") if part]
        numbers = []
        for part in parts:
            try:
                numbers.append(float(part.replace(",", ".")))
            except ValueError:
                return None
        return float(sum(numbers) / len(numbers)) if numbers else None
    return parse_numeric(text)


def parse_blood_pressure(value):
    text = clean_text(value)
    if not text:
        return None, None, None
    text = text.replace(" ", "")
    match = re.match(r"^(\d{2,3})/(\d{2,3})$", text)
    if not match:
        return None, None, None
    systolic = int(match.group(1))
    diastolic = int(match.group(2))
    if 160 <= systolic <= 180:
        attr = 3
    elif systolic >= 140 or diastolic >= 90:
        attr = 2
    else:
        attr = 1
    return attr, systolic, diastolic


def transform_gravida(value):
    number = parse_numeric(value)
    if number is None:
        return None, None
    return (1 if number == 1 else 2), float(number)


def transform_djj(value):
    number = parse_numeric(value)
    if number is None:
        return None, None
    if 120 <= number <= 160:
        attr = 0
    else:
        attr = 1
    return attr, float(number)


def transform_kpd(value):
    text = clean_text(value).lower()
    if not text:
        return None
    if text == "tidak":
        return 1
    if text == "ada":
        return 0
    return None


def standardize_columns(df):
    renamed = []
    for column in df.columns:
        clean_column = clean_text(column)
        renamed.append(COLUMN_ALIASES.get(clean_column, clean_column or column))
    df = df.copy()
    df.columns = renamed
    for column in ["No", "Nomor RM", *FEATURE_COLUMNS, "Tindakan"]:
        if column not in df.columns:
            df[column] = np.nan
    return df


def transform_rows(raw_df):
    records = []
    original_columns = list(raw_df.columns)
    for _, row in raw_df.iterrows():
        record = {column: row.get(column, np.nan) for column in original_columns}
        reasons = []

        no_value = parse_numeric(row.get("No"))
        nomor_rm = normalize_rm(row.get("Nomor RM"))
        usia_ibu = parse_numeric(row.get("Usia Ibu"))
        usia_kehamilan = parse_range_numeric(row.get("Usia Kehamilan"))
        gravida_attr, gravida_raw = transform_gravida(row.get("Gravida"))
        tekanan_attr, systolic, diastolic = parse_blood_pressure(row.get("Tekanan Darah"))
        djj_attr, djj_raw = transform_djj(row.get("Denyut Jantung Janin"))
        nilai_afi = parse_numeric(row.get("Nilai AFI"))
        kpd_attr = transform_kpd(row.get("Kejadian KPD"))
        tindakan_val = remove_mow_text(row.get("Tindakan"))

        if no_value is None:
            reasons.append("No kosong")
        if not nomor_rm:
            reasons.append("Nomor RM kosong")
        if usia_ibu is None:
            reasons.append("Usia Ibu tidak valid/kosong")
        if usia_kehamilan is None:
            reasons.append("Usia Kehamilan tidak valid/kosong")
        if gravida_attr is None:
            reasons.append("Gravida tidak valid/kosong")
        if tekanan_attr is None:
            reasons.append("Tekanan Darah tidak valid/kosong")
        if djj_attr is None:
            reasons.append("Denyut Jantung Janin tidak valid/kosong")
        if nilai_afi is None:
            reasons.append("Nilai AFI tidak valid/kosong")
        if kpd_attr is None:
            reasons.append("Kejadian KPD tidak valid/kosong")

        record["No"] = int(no_value) if no_value is not None else np.nan
        record["Nomor RM"] = nomor_rm or np.nan
        record["Usia Ibu"] = usia_ibu if usia_ibu is not None else np.nan
        record["Usia Kehamilan"] = usia_kehamilan if usia_kehamilan is not None else np.nan
        record["Gravida"] = gravida_attr if gravida_attr is not None else np.nan
        record["Tekanan Darah"] = tekanan_attr if tekanan_attr is not None else np.nan
        record["Denyut Jantung Janin"] = djj_attr if djj_attr is not None else np.nan
        record["Nilai AFI"] = nilai_afi if nilai_afi is not None else np.nan
        record["Kejadian KPD"] = kpd_attr if kpd_attr is not None else np.nan
        record["Tindakan"] = remove_mow_text(row.get("Tindakan")) or np.nan
        record["Gravida Asli"] = gravida_raw if gravida_raw is not None else np.nan
        record["Denyut Jantung Janin Asli"] = djj_raw if djj_raw is not None else np.nan
        record["Sistolik"] = systolic if systolic is not None else np.nan
        record["Diastolik"] = diastolic if diastolic is not None else np.nan
        record["Status Validasi"] = "valid" if not reasons else "invalid"
        record["Alasan Invalid"] = "; ".join(reasons)
        records.append(record)
    return pd.DataFrame(records)


def normalize_dataframe(valid_df, min_values, max_values):
    normalized = valid_df.copy()
    for column in FEATURE_COLUMNS:
        min_value = min_values[column]
        max_value = max_values[column]
        if max_value == min_value:
            normalized[column] = 0.0
        else:
            normalized[column] = (normalized[column].astype(float) - min_value) / (max_value - min_value)
    return normalized


def compute_distances(data, centers):
    distances = np.linalg.norm(data[None, :, :] - centers[:, None, :], axis=2)
    return np.maximum(distances, 1e-10)


def update_membership(distances, m=2.0):
    exponent = 2.0 / (m - 1.0)
    membership = np.zeros_like(distances)
    for sample_index in range(distances.shape[1]):
        sample_distances = distances[:, sample_index]
        zero_mask = np.isclose(sample_distances, 0.0)
        if zero_mask.any():
            membership[zero_mask, sample_index] = 1.0 / zero_mask.sum()
            continue
        ratios = (sample_distances[:, None] / sample_distances[None, :]) ** exponent
        membership[:, sample_index] = 1.0 / ratios.sum(axis=1)
    return membership


def predict_action(cluster_number, usia_ibu_asli=None, gravida_asli=None):
    if cluster_number == 1:
        return "Konservatif / Induksi / Partus spontan"
    if cluster_number == 2:
        return "NST / Induksi / SC / SCTP"
    return "SC cito / SC"



In [3]:
model = json.loads((ARTIFACT_DIR / "fcm_model.json").read_text(encoding="utf-8"))
raw_df = pd.read_csv(RAW_TEST_PATH)
raw_df = standardize_columns(raw_df)
transformed_df = transform_rows(raw_df)

valid_df = transformed_df.loc[transformed_df["Status Validasi"] == "valid"].copy()
invalid_df = transformed_df.loc[transformed_df["Status Validasi"] == "invalid"].copy()

invalid_export_columns = list(raw_df.columns) + [
    "Gravida Asli",
    "Denyut Jantung Janin Asli",
    "Sistolik",
    "Diastolik",
    "Status Validasi",
    "Alasan Invalid",
]
invalid_export_columns = [column for column in invalid_export_columns if column in invalid_df.columns]
invalid_df[invalid_export_columns].to_csv(TESTING_DIR / "output_testing_invalid.csv", index=False, encoding="utf-8-sig")

normalized_valid = normalize_dataframe(valid_df, model["min_values"], model["max_values"])
output_preprocessing = normalized_valid[["No", "Nomor RM", *FEATURE_COLUMNS]].copy().round(6)
output_preprocessing["No"] = output_preprocessing["No"].astype(int)
output_preprocessing["Nomor RM"] = output_preprocessing["Nomor RM"].astype(str)
output_preprocessing.to_csv(TESTING_DIR / "output_testing_preprocessing.csv", index=False, encoding="utf-8-sig")

base_df = output_preprocessing[["No", "Nomor RM"]].copy()
data = output_preprocessing[FEATURE_COLUMNS].to_numpy(dtype=float)
centers = np.array(model["centers"], dtype=float)
distances = compute_distances(data, centers)
membership = update_membership(distances, m=float(model["params"]["m"]))
squared = membership ** 2
labels = [CLUSTER_LABELS[index] for index in np.argmax(membership, axis=0)]

output_keanggotaan = base_df.copy()
output_keanggotaan[["μ1", "μ2", "μ3"]] = membership.T
output_keanggotaan = output_keanggotaan.round(6)
output_keanggotaan.to_csv(TESTING_DIR / "output_testing_keanggotaan.csv", index=False, encoding="utf-8-sig")

output_euclidean = base_df.copy()
output_euclidean[["d1", "d2", "d3"]] = distances.T
output_euclidean = output_euclidean.round(6)
output_euclidean.to_csv(TESTING_DIR / "output_testing_euclidean.csv", index=False, encoding="utf-8-sig")

prediksi_tindakan = [
    predict_action(index + 1, valid_df.iloc[row_index]["Usia Ibu"], valid_df.iloc[row_index]["Gravida Asli"])
    for row_index, index in enumerate(np.argmax(membership, axis=0))
]

output_final = base_df.copy()
output_final[["K1", "K2", "K3"]] = membership.T
output_final[["μ1²", "μ2²", "μ3²"]] = squared.T
output_final["Hasil"] = labels
output_final["Prediksi Tindakan"] = prediksi_tindakan
output_final = output_final.round(6)
output_final.to_csv(TESTING_DIR / "output_testing_final.csv", index=False, encoding="utf-8-sig")

print(f"Data testing valid   : {len(valid_df)}")
print(f"Data testing invalid : {len(invalid_df)}")
print("Output testing berhasil diperbarui.")
output_final.head()

Data testing valid   : 15
Data testing invalid : 0
Output testing berhasil diperbarui.


,No,Nomor RM,K1,K2,K3,μ1²,μ2²,μ3²,Hasil,Prediksi Tindakan
0,241,083790,0.507989,0.242870,0.249140,0.258053,0.058986,0.062071,Mild,Konservatif / Induksi / Partus spontan
1,242,084788,0.073479,0.041499,0.885022,0.005399,0.001722,0.783263,Severe,SC cito / SC
2,243,078086,0.155026,0.751116,0.093858,0.024033,0.564176,0.008809,Moderate,NST / Induksi / SC / SCTP
3,244,084816,0.133522,0.782344,0.084134,0.017828,0.612062,0.007079,Moderate,NST / Induksi / SC / SCTP
4,245,084195,0.136780,0.774696,0.088524,0.018709,0.600154,0.007836,Moderate,NST / Induksi / SC / SCTP
